# exp-01 Pre-resolved places in the input

**Question.** If code finds and looks up the place names in the raw text before the LLM call and puts the
results in the input, how do LLM steps, `resolve_place` calls, tokens, latency and pass rate change?

**Dataset.** `data/examples.json`, 300 single-turn examples. Step 1 is free; step 2 calls the LLM and runs only
when `RUN_PAID = True`.

In [ ]:
# Setup: works whether the notebook's folder or the repo root is the working directory.
import json, sys
from pathlib import Path
HERE = Path.cwd()
if HERE.name != "exp01_preresolve":
    HERE = next(p for p in [Path.cwd() / "eng-experiments" / "exp01_preresolve", Path.cwd() / "exp01_preresolve"] if p.exists())
sys.path[:0] = [str(HERE), str(HERE.parent)]
import os; os.environ.setdefault("TRACE_PRINT", "0")   # no per-span printing during runs

import pandas as pd
from IPython.display import Markdown, display
import common
import matchers, offline, run
pd.set_option("display.max_colwidth", 120)

## Settings

In [ ]:
RUN_PAID = False          # True = really call the LLM (costs money)
FIRST = None              # e.g. 3 for a smoke run; None = all 300 examples
THRESHOLD = [85, 90, 95]  # arm 3's fuzzy score(s); one arm 3 per value (from step 1)
MODEL = common.DEFAULT_MODEL
examples = common.load_examples(first=FIRST)
print(len(examples), "examples; model", MODEL)

## Step 1: matcher check (no LLM)

Code alone vs the gold `mentions`. Also rewrites the step-1 tables in `README.md`.

In [ ]:
offline.main()
res = json.loads((HERE / "results" / "offline.json").read_text())
pd.DataFrame(res["overall"]).set_index("matcher")

In [ ]:
# By tag
pd.concat({tag: pd.DataFrame(rows).set_index("matcher")[["precision", "recall", "false_positives", "wrong_ok",
                                                          "examples_all_covered"]]
           for tag, rows in res["by_tag"].items()})

In [ ]:
# False positives and wrong `ok`, per matcher
for name, L in res["lists"].items():
    print(name, "\n  false positives:", L["false_positives"], "\n  wrong ok:", L["wrong_ok"])

In [ ]:
# What the LLM would get for one example (arm 2 vs arm 3)
t = examples[0]["text"]
print(t)
print("exact:", matchers.exact(t))
print("exact+fuzzy:", matchers.exact_fuzzy(t, THRESHOLD[1]))

## Step 2: the arms

In [ ]:
variants = run.arms(THRESHOLD)
pd.DataFrame([{"arm": v.name, "prompt": Path(v.prompt_file).stem, "input": v.notes} for v in variants]).set_index("arm")

In [ ]:
turns = len(examples) * len(variants)
print(f"{turns} agent turns, about ${turns * run.USD_PER_TURN:.2f}")

## Step 2: run (paid)

In [ ]:
if RUN_PAID:
    assert THRESHOLD, "set THRESHOLD first"
    rows = common.run_examples("exp01", run.arms(THRESHOLD), examples, MODEL, HERE / "results")
else:
    print("RUN_PAID is False: nothing was called.")

## Step 2: results (latest run in `results/`)

Also rewrites the step-2 tables in `README.md`.

In [ ]:
path = common.latest(HERE / "results", "exp01-*.json")
if path:
    data = json.loads(path.read_text())
    rows, arm_names = data["rows"], [a["name"] for a in data["meta"]["arms"]]
    display(common.frame(rows, arm_names))
    display(Markdown(run.report(path)))
else:
    print("No results yet.")

In [ ]:
# Per example: steps and pass per arm, and the examples where the arms disagree on pass
if path:
    df = pd.DataFrame(rows)
    wide = df.pivot(index="example", columns="arm", values=["pass", "steps", "resolve_calls", "seconds"])
    differ = wide["pass"][wide["pass"].nunique(axis=1) > 1]
    display(wide.head(20))
    display(differ.join(df.drop_duplicates("example").set_index("example")["text"]))